In [ ]:
import os
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from scipy import sparse
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import sys
import wandb
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import re
import scipy

In [ ]:
sys.path.append("/content/drive/MyDrive/Genre_Detector")
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 72.0 MB/s eta 0:00:00


In [ ]:
from models import train_one_epoch, evaluate, MLP_basic, biLSTM
from embeddings import get_features

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        if scipy.sparse.issparse(X):
            self.X = X.tocsr()
        else:
            self.X = np.asarray(X)
        self.y = np.asarray(y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x_item = self.X[idx]
        if scipy.sparse.issparse(x_item):
            x_item = x_item.toarray().squeeze()

        x_dtype = torch.long if self.is_lstm else torch.float32
        return (
            torch.tensor(x_item, dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

In [ ]:
def run_experiment(df, y, confing_path='./default_conf_experiment'):

    # Pełna, bezpieczna baza parametrów
    default_config = {
        "project_name": "Genre_Detector",
        "experiment_name": "mlp_baseline",
        "embedding_method": "w2v",
        "embedding_method_name": "word2vec-google-news-300",
        "max_features": 300,
        "data_size": 10000,
        "test_size": 0.2,
        "random_state": 42,
        "batch_size": 64,
        "lr": 1e-3,
        "epochs": 10,
        "threshold": 0.5,
        "use_pos_weight": True
    }

    if config_path is not None:
        with open(config_path, "r", encoding="utf-8") as conf_file:
            user_config = json.load(conf_file)
        default_config.update(user_config)

    wandb.init(
        project=default_config["project_name"],
        name=default_config["experiment_name"],
        config=default_config
    )
    cfg = wandb.config

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running experiment on device: {device}")

    X = get_features(df, embedding_method=cfg.embedding_method, max_features=cfg.max_features)

    X = X[:cfg.data_size]
    y = y[:cfg.data_size]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_size, random_state=cfg.random_state
    )

    train_dataset = TextDataset(X_train, y_train)
    test_dataset = TextDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)

    input_dim = X_train.shape[1]
    num_classes = y_train.shape[1]

    embedding_model = None
    if cfg.embedding_method == 'w2v': 
        embedding_model = gensim.downloader.load(cfg.embedding_method_name)

    if "mlp" in cfg.experiment_name.lower():
        input_dim = X_train.shape[1]
        model = MLP_basic(
            input_dim=input_dim, 
            num_classes=num_classes
        ).to(device)

    elif "bilstm" in cfg.experiment_name.lower():
        if embedding_model is None:
            raise ValueError("BiLSTM requires a pretrained Word2Vec model.")
        model = BiLSTM(
            pretrained_w2v=embedding_model,
            embedding_dim=embedding_model.vector_size,
            hidden_dim=128,
            num_classes=num_classes
        ).to(device)
    else:
        raise ValueError(f"Unknown experiment name: {cfg.experiment_name}")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    if cfg.use_pos_weight:
        y_train_np = np.asarray(y_train)
        pos_counts = y_train_np.sum(axis=0)
        neg_counts = len(y_train_np) - pos_counts
        pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(device)
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(cfg.epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_f1 = evaluate(model, test_loader, criterion, device, threshold=cfg.threshold)

        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val micro-F1: {val_f1:.4f}")

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_micro_f1": val_f1
        })

    wandb.finish()

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/data_goodreads.csv")

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return None

def parse_genres(x):
    if not isinstance(x, str):
        return []
    return re.findall(r'"(.*?)"', x)

def encode_genres(genres):
  vec = np.zeros(30, dtype=np.float32)
  for g in genres:
      if g in genre_to_idx:
          vec[genre_to_idx[g]] = 1.0
  return vec

bad_rows = df[df["genres"].apply(lambda x: safe_eval(x) is None)]
df["genres"] = df["genres"].apply(parse_genres)
top30 = df["genres"].explode().value_counts().head(30)
top30 = df["genres"].explode().value_counts().head(30).index.tolist()
genre_to_idx = {g: i for i, g in enumerate(top30)}
y = np.stack(df["genres"].apply(encode_genres))

In [ ]:
w2v_config = {
    "project_name": "Genre_Detector",
    "experiment_name": "mlp_baseline",
    "embedding_method": "w2v",
    "embedding_method_name": "word2vec-google-news-300", 
    "max_features": 300,
    "batch_size": 64,
    "data_size": 10000,
    "lr": 1e-3,
    "epochs": 5
}

wandb.login()
run_experiment(df, y, config_input=w2v_config)